# 5b (Capstone) - The Graph-Mined Base: `CatalogBaseSource`

Chapter 5 gives three base sources. The capstone uses `SeedCaseSource` --- twenty *user-authored* cases --- because it tests an agent's trajectory, which needs hand-labeled per-component truth. This companion shows the **third** source, `CatalogBaseSource`, which *mines* test questions from a trained GMS store and reads their ground truth straight from the graph.

One thing to be precise about, because it is easy to overstate: **you still choose which question types to generate.** `CatalogBaseSource` is not a system that inspects the graph and decides what to ask --- you name the categories (`exact_recall`, `counting`, ...), and the generators then produce instances and answers for *those* categories. What is automatic is narrower: generating the question instances, deriving their answers from the graph, and **dropping a requested category to zero when the graph has no structure to fill it**. Selection is yours; the graph only vetoes.

It is shown here on the capstone's own policy store for continuity, but the point is general: an application with a trained graph but **no** labeled cases can generate a provably-correct base by selecting the categories its graph supports. (The capstone uses the same underlying knowlytix generators on its substrate side to build its answer key; here we drive them through the gms-testing wrapper for teaching.)

## When to use it

| | `SeedCaseSource` (Chapter 5 notebook) | `CatalogBaseSource` (this notebook) |
|---|---|---|
| Which questions / types | **user-specified** (you write each case) | **user-specified** (you name the category types) |
| Where the question text comes from | hand-authored messages | generated from the chosen categories over the graph |
| Where ground truth comes from | author + graph-read fact key | read from the graph, provably correct |
| What it tests | agent trajectory, per-tool | knowledge recall / aggregation / reasoning |
| Needs labeled cases? | yes | **no** --- only a trained store |

Note that **selection is the user's in both** --- the difference is not "user vs automatic," it is *where the questions and answers come from*. Reach for `CatalogBaseSource` when you have a store and want coverage of the taxonomy's *recall and reasoning* categories without writing any labels.

**What the next cell does** — imports the framework, loads the capstone's trained policy store, and lists the categories that have a working generator:

1. **Wire up imports.** Adds the tutorial root to `sys.path` and imports the `gmstest` wrapper as `gt`, plus `DocGMSConfig` and `GMSExpertStore` from `knowlytix.knowledge.query`.
2. **Find the store on disk.** Walks parent directories from the cwd until it locates `data/gms_policy_store_geode`, the GEODE policy store.
3. **Load the store.** Builds a `GMSExpertStore` with `ingest_mode='regex'` on CPU and asserts `store.load()` succeeds, printing the entity, relation and triple counts.
4. **List implemented categories.** Loads the base catalog with `gt.Catalog.load()` and keeps the names of every base whose `status == 'implemented'` — the categories that actually have a generator.

In [ ]:
import sys, pathlib, torch
from collections import Counter
sys.path.insert(0, str(pathlib.Path.cwd().parent / "code"))   # gmstest
import gmstest as gt
from knowlytix.knowledge.query import DocGMSConfig, GMSExpertStore

ROOT = pathlib.Path.cwd().parents[1] / 'beyond-prompt-and-pray' / 'code'
while not (ROOT / 'data' / 'gms_policy_store_geode').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
store = GMSExpertStore(
    DocGMSConfig(store_path=str(ROOT / 'data' / 'gms_policy_store_geode'), ingest_mode='regex'),
    device=torch.device('cpu'))
assert store.load()

cat = gt.Catalog.load()
implemented = [b.name for b in cat.bases.values() if b.status == 'implemented']
print('categories with a working generator:', implemented)

## Mine the store

We resolve a suite of the implemented categories and let `CatalogBaseSource` run each category's generator over the policy graph. Each generator emits questions *and* their ground truth; we cap it at three per category to keep the output readable.

**What the next cell does** — resolves a suite of all implemented categories, mines questions from the store, and prints one sample per category:

1. **Resolve the suite.** `gt.resolve(cat, implemented, [], mode='cross')` expands the requested base names into specs (no factors here) and returns a `ResolvedSuite`.
2. **Mine the graph.** `gt.CatalogBaseSource(store, max_per_category=3).items(suite)` instantiates each category's knowlytix generator, calls `gen.generate(graph)`, samples at most three per category, and wraps each `GeneratedQuestion` as a `QAItem` carrying its `query`, graph-read `answer` and `base`.
3. **Tally per category.** A `Counter` over `it.base` reports how many questions each category produced; `mined_zero` lists requested categories that produced none.
4. **Show one example each.** For every category that fired, prints the first item's answer type, truncated question text and graph-derived ground truth.

In [ ]:
suite = gt.resolve(cat, implemented, [], mode='cross')
items = gt.CatalogBaseSource(store, max_per_category=3).items(suite)

counts = Counter(it.base for it in items)
print(f'{len(items)} questions mined from the policy store')
print('per category :', dict(counts))
mined_zero = [c for c in implemented if c not in counts]
print('mined zero   :', mined_zero, '(no qualifying graph structure)\n')

for base in counts:
    it = next(i for i in items if i.base == base)
    print(f"[{base}]  ({it.answer_type})")
    print('  Q:', it.query[:88])
    print('  ground truth:', it.answer, '\n')

## The ground truth is read from the graph

Because the answer is computed by walking the store, it is provably correct --- no annotator, no second opinion. Two things follow. First, a generator only emits where the graph supports it: on this clean oracle `contradiction` mines nothing, because there are no conflicting facts to find, and `conditional_rule` finds no threshold structure it recognizes. Second, any mined answer can be independently re-derived from the store. We check the `cross_reference` answer by recomputing the set intersection ourselves.

**What the next cell does** — independently re-derives the `cross_reference` answer straight from the store to prove it is graph-read:

1. **Pull the question and its relations.** Selects the `cross_reference` item and reads the two relation names it joins from `xr.metadata['relation_1']` and `relation_2`.
2. **Recompute by hand.** A `heads(r)` helper calls `store.query_triples(relation=r)` and collects the head entities; intersecting `heads(r1) & heads(r2)` reconstructs the answer set.
3. **Compare.** Prints the mined set against the recomputed set and confirms `recomputed == set(xr.answer)` — the mined answer matches an independent re-derivation.

In [ ]:
xr = next(it for it in items if it.base == 'cross_reference')
r1, r2 = xr.metadata['relation_1'], xr.metadata['relation_2']

# re-derive the answer straight from the store: heads of r1 intersected with heads of r2
heads = lambda r: {h for h, _, _ in store.query_triples(relation=r)}
recomputed = heads(r1) & heads(r2)

print(f'question  : entities in both {r1!r} and {r2!r}')
print('mined     :', sorted(xr.answer))
print('recomputed:', sorted(recomputed))
print('match     :', recomputed == set(xr.answer))

So a user *can* generate a base automatically from the policy GMS, with no hand labels --- and that is exactly the machinery the capstone uses on its substrate side to build the answer key the seed cases are scored against (Chapter 4, notebook 12.4). The capstone calls those knowlytix generators directly; here the gms-testing `CatalogBaseSource` wrapper exposes the same path for any application that wants a graph-mined base of its own.

## You choose the categories; the graph only vetoes

To make the division of labor concrete: we *request* two categories --- one the store can fill and one it cannot. The framework does not pick for us; it runs the generator we asked for and simply returns zero questions for the one with no qualifying structure.

The example also shows a subtler point. Passing a category's capability check is **necessary but not sufficient**. `contradiction` requires only `triples`, which this store plainly has (`counting` and `cross_reference` need the same capability and both fired). It still mines nothing --- because the *structure* it hunts for, a pair of conflicting facts, is absent from a clean single-source oracle. So even an up-front `requires`-vs-capability check could not predict the zero; only running the generator reveals it.

**What the next cell does** — requests two categories, one fillable and one not, to show the user selects and the graph only vetoes:

1. **Name the request.** `requested = ['exact_recall', 'contradiction']` — one category the store supports and one it does not.
2. **Mine and count.** Resolves the pair and runs `CatalogBaseSource(...).items(...)`, tallying results per `it.base` into `got`.
3. **Report the veto.** Prints each requested category with its count, flagging `contradiction` as dropped to zero because no qualifying structure exists.
4. **Show necessary-not-sufficient.** Prints `cat.bases['contradiction'].requires` (`['triples']`) against the fact that the store has triples yet still mined zero — passing the capability check does not guarantee questions.

In [ ]:
# We name the categories -- one the store supports, one it does not.
requested = ['exact_recall', 'contradiction']
got = Counter(it.base for it in
              gt.CatalogBaseSource(store, max_per_category=3).items(
                  gt.resolve(cat, requested, [], mode='cross')))

print('selection is ours; the graph vetoes what it cannot fill:')
for c in requested:
    n = got.get(c, 0)
    tag = '' if n else '   <- requested, dropped to 0 (no qualifying structure)'
    print(f'  requested {c:14s} -> {n} questions{tag}')

# capability is necessary, not sufficient: contradiction passes the requires check
# (the store has triples) yet still mines nothing, because the structure is absent.
print(f"\ncontradiction requires {cat.bases['contradiction'].requires}; "
      f"store has triples (counting fired) -- yet contradiction mined {got.get('contradiction', 0)}")

## One fact, many question types

A graph item is not owned by one category. Every generator whose pattern it matches turns it into a question, independently and with no de-duplication --- so the same fact yields several Q&A across categories. The overdraft fee (`overdraft -> has_fee_amount -> 35`) is a clean example: it is the value `exact_recall` recalls, one of the members `counting` tallies, an element of the set `cross_reference` intersects, and an entry `multi_hop` ranks in the fee schedule. One fact, four question shapes --- which for a test suite is coverage, not redundancy.

**What the next cell does** — mines a wider sample and pulls the one overdraft-fee fact as it surfaces under four different categories:

1. **Mine broadly.** Runs `CatalogBaseSource(store, max_per_category=50).items(...)` over the four producing categories to get a fuller corpus into `wide`.
2. **Filter to the fee fact.** A `touches_fee(it)` predicate lowercases the item's query, answer and metadata and matches on `'fee'` or `'overdraft'`.
3. **One fact, four shapes.** For each of `exact_recall`, `counting`, `cross_reference` and `multi_hop`, picks the first matching item and prints its question and answer — the same fee fact reused as four question types with no de-duplication.

In [ ]:
# Mine a wider sample, then pull the questions that touch the overdraft fee /
# the has_fee_amount relation it lives in -- the SAME fact across categories.
wide = gt.CatalogBaseSource(store, max_per_category=50).items(
    gt.resolve(cat, ['exact_recall', 'counting', 'cross_reference', 'multi_hop'], [], mode='cross'))

def touches_fee(it):
    blob = (it.query + ' ' + str(it.answer) + ' ' + str(it.metadata)).lower()
    return 'fee' in blob or 'overdraft' in blob

print('the overdraft fee fact surfaces as a different question in each category:\n')
fee_q = {}
for base in ['exact_recall', 'counting', 'cross_reference', 'multi_hop']:
    it = next(i for i in wide if i.base == base and touches_fee(i))
    fee_q[base] = it
    print(f'[{base}]')
    print('  Q:', it.query[:84])
    print('  A:', str(it.answer)[:68], '\n')

## The functionality check: symbolic vs geometric

A mined question is only admitted if its answer can be *recovered* from the store. There are two ways to recover it, and the store-based generators support both:

- **Symbolic** (the default): re-run the generator's answer function against the store's index and confirm it equals the stated ground truth. This is what guards the questions above.
- **Geometric** (`geometric=True`): verify the answer a *second* way against the trained GMS geometry. Each answer fact triple must (a) be more plausible than the relation's non-asserted tails, and (b) sit under that relation's **learned admissible radius** `cap_radius(relation)` --- the calibrated operating point the cap geometry fitted during training, never a hand-set threshold. A question survives only when the symbolic oracle and the geometry agree.

Every admitted question carries an auditable `_recovery` tag (`two_way`, or `symbolic:<reason>` for an aggregation like a count that has no fact triple), so nothing is verified silently.

**What the next cell does** — runs the `exact_recall` generator's functionality check both symbolically and geometrically and reports the recovery split:

1. **Import the generator.** Pulls `ExactRecallGenerator` from `knowlytix.benchmark.eval.generators.exact_recall`.
2. **Symbolic pass.** `ExactRecallGenerator(geometric=False).generate(store)` admits questions whose `gms_answer_fn` recomputes to the stated ground truth.
3. **Two-way pass.** `ExactRecallGenerator(geometric=True).generate(store)` additionally requires each answer triple to be geometrically recovered before admitting.
4. **Report the audit.** Prints the admitted counts, the `recovery_stats` split (`two_way` / `symbolic` / `rejected`), the calibrated `last_recovery_note` (the learned `cap_radius`) and the per-question `_recovery` tags.

In [ ]:
# the store-based eval generators carry the functionality check; run it both ways
from knowlytix.benchmark.eval.generators.exact_recall import ExactRecallGenerator

sym = ExactRecallGenerator(geometric=False).generate(store)   # symbolic recompute only
gg  = ExactRecallGenerator(geometric=True)
geo = gg.generate(store)                                       # symbolic AND geometric
print(f'exact_recall admitted -- symbolic-only: {len(sym)}   two-way (geometric): {len(geo)}')
print(f'recovery split        : {dict(gg.recovery_stats)}')
print(f'calibrated cut        : {gg.last_recovery_note}')
print(f'sample recovery tags  : {[q.metadata["_recovery"] for q in geo[:3]]}')

On a clean oracle the two checks agree, so both admit the same 26 facts. The geometric check earns its keep where they *disagree* --- a question whose stated answer is symbolically constructible but geometrically implausible. We forge one: the overdraft-fee question with its answer triple corrupted to the *wire* fee (\$45). The symbolic recompute still returns the true value (the question is "valid" on the index), but the geometry refuses the implausible triple.

**What the next cell does** — forges a true and a corrupted question and shows the geometric check rejects what the symbolic check admits:

1. **Read the true fact.** Finds the overdraft ENM key in `store.enm._store` and reads its scalar `value`; `answer_fn` returns that value from the store.
2. **Build two questions.** `true_q` is a faithful `GeneratedQuestion`; `bad_q` reuses the same `answer_fn` (so it recomputes correctly) but declares an implausible `answer_triples` of `('overdraft', 'has_fee_amount', '45.0')` — the wire fee, not overdraft's.
3. **Validate both ways.** Calls `_validate(store, q)` on a `geometric=False` and a `geometric=True` generator for each question.
4. **Show the disagreement.** Prints the table: both questions pass symbolic, but only the geometry refuses `bad_q` because its declared triple sits outside the learned cap radius.

In [ ]:
from knowlytix.benchmark.eval.generators.base import GeneratedQuestion

key = next(k for k in store.enm._store if 'overdraft' in k.id.lower())
val = float(store.enm._store[key].value.item())
answer_fn = lambda s, k=key: float(s.enm._store[k].value.item())

true_q = GeneratedQuestion('demo-true', 'exact_recall', 'What is the overdraft fee?',
                           answer_fn, val, '', 'float', metadata={'enm_id': key.id})
# symbolically valid (same answer_fn) but its DECLARED triple is the wire fee, not overdraft's
bad_q  = GeneratedQuestion('demo-bad', 'exact_recall', 'What is the overdraft fee?',
                           answer_fn, val, '', 'float',
                           metadata={'answer_triples': [('overdraft', 'has_fee_amount', '45.0')]})

g_sym, g_geo = ExactRecallGenerator(geometric=False), ExactRecallGenerator(geometric=True)
print(f"{'question':16s}{'symbolic':>12s}{'geometric':>12s}")
print(f"{'true fact':16s}{str(g_sym._validate(store, true_q)):>12s}{str(g_geo._validate(store, true_q)):>12s}")
print(f"{'implausible':16s}{str(g_sym._validate(store, bad_q)):>12s}{str(g_geo._validate(store, bad_q)):>12s}"
      "   <- geometry rejects what symbolic admits")

## Self-check

The asserts below pin every number this notebook claims --- six implemented categories, twelve questions mined from the four that the clean store supports, every one carrying graph-read ground truth, and a `cross_reference` answer that re-derives exactly from the store. Run the notebook end to end and it fails loud if any of these drift, so the illustration stays reproducible rather than quietly going stale.

**What the next cell does** — asserts every number the notebook claims so the illustration fails loud if anything drifts:

1. **Pin the catalog and mine counts.** Asserts six implemented categories, twelve mined items all answered, the four producing categories, and the two `mined_zero` categories.
2. **Pin the re-derivation.** Asserts `recomputed == set(xr.answer)` so the graph-read answer matches.
3. **Pin selection-vs-veto.** Asserts `exact_recall` fired and `contradiction` was vetoed, and that the fee fact appears in all four producing categories.
4. **Pin the functionality check.** Asserts symbolic and geometric admit equal counts with all `two_way`, both checks admit the true fact, and only the geometry rejects the corrupted one; prints the closing OK line.

In [ ]:
assert len(implemented) == 6                                  # six categories have generators
assert len(items) == 12 and all(it.answer is not None for it in items)   # all mined, all answered
assert set(counts) == {'exact_recall', 'counting', 'cross_reference', 'multi_hop'}
assert set(mined_zero) == {'conditional_rule', 'contradiction'}   # clean store -> nothing to mine
assert recomputed == set(xr.answer)                          # graph-read answer re-derives exactly
# selection is user-specified; the graph only vetoes the unfit
assert got.get('exact_recall', 0) > 0 and got.get('contradiction', 0) == 0
# one fact, many categories: the fee fact surfaces in all four producing categories
assert set(fee_q) == {'exact_recall', 'counting', 'cross_reference', 'multi_hop'}
# functionality check: symbolic and geometric agree on true facts; geometry rejects
# an implausible triple that symbolic admits
assert len(sym) == len(geo) and gg.recovery_stats['two_way'] == len(geo)
assert g_sym._validate(store, true_q) and g_geo._validate(store, true_q)       # both admit truth
assert g_sym._validate(store, bad_q) and not g_geo._validate(store, bad_q)     # geometry alone rejects
print('OK - 5b capstone: a provably-correct base mined from the policy graph, no labels needed')